In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week6-lesson-3"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
## create dataframe from local array

In [3]:
data = [("Spring", 12.3),("Summer", 10.5),("Autumn", 8.2),("Winter", 15.1)]

In [4]:
data_df = spark.createDataFrame(data).toDF("season","windspeed")

In [5]:
data_df.show()

+------+---------+
|season|windspeed|
+------+---------+
|Spring|     12.3|
|Summer|     10.5|
|Autumn|      8.2|
|Winter|     15.1|
+------+---------+



In [6]:
data_df.printSchema()

root
 |-- season: string (nullable = true)
 |-- windspeed: double (nullable = true)



In [7]:
#Consider the library management dataset located at the following path(/public/trendytech/datasets/library_data.json). Using PySpark, load thedata into a Dataframe and enforce schema using StructType.

In [8]:
!hadoop fs -head /public/trendytech/datasets/library_data.json

{"library_name": "Central Library","location": "City Center","books": [{"book_id": "B001","book_name": "The Great Gatsby","author": "F. Scott Fitzgerald","copies_available": 5},{"book_id": "B002","book_name": "To Kill a Mockingbird","author": "Harper Lee","copies_available": 3}],"members": [{"member_id": "M001","member_name": "John Smith","age": 28,"books_borrowed": ["B001"]},{"member_id": "M002","member_name": "Emma Johnson","age": 35,"books_borrowed": []}]},
{"library_name": "Community Library","location": "Suburb","books": [{"book_id": "B003","book_name": "1984","author": "George Orwell","copies_available": 2},{"book_id": "B004","book_name": "Pride and Prejudice","author": "Jane Austen","copies_available": 4}],"members": [{"member_id": "M003","member_name": "Michael Brown","age": 42,"books_borrowed": ["B003","B004"]},{"member_id": "M004","member_name": "Sophia Davis","age": 31,"books_borrowed": ["B004"]}]}


## StructType Schema for Nested JSON with ARRAYS

In [9]:
book_schema = StructType ([
    StructField("book_id",StringType()),
    StructField("book_name",StringType()),
    StructField("author",StringType()),
    StructField("copies_available",LongType()),    
])

members_schema = StructType ([
    StructField("member_id",StringType()),
    StructField("member_name",StringType()),
    StructField("age",LongType()),
    StructField("books_borrowed",ArrayType(StringType())),    
])

jsonStruct = StructType ([
    StructField("library_name",StringType()),
    StructField("location",StringType()),
    StructField("books",ArrayType(book_schema)),
    StructField("members",ArrayType(members_schema)),
])

In [10]:
json_df = spark.read.format('json').schema(jsonStruct).load('/public/trendytech/datasets/library_data.json')

In [11]:
json_df.show(truncate=False)

+-----------------+-----------+------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------+
|library_name     |location   |books                                                                                           |members                                                                    |
+-----------------+-----------+------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------+
|Central Library  |City Center|[{B001, The Great Gatsby, F. Scott Fitzgerald, 5}, {B002, To Kill a Mockingbird, Harper Lee, 3}]|[{M001, John Smith, 28, [B001]}, {M002, Emma Johnson, 35, []}]             |
|Community Library|Suburb     |[{B003, 1984, George Orwell, 2}, {B004, Pride and Prejudice, Jane Austen, 4}]                   |[{M003, Michael Brown, 42, [B003, B004]}, {M004, Sop

In [12]:
json_df.printSchema()

root
 |-- library_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- books: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- book_id: string (nullable = true)
 |    |    |-- book_name: string (nullable = true)
 |    |    |-- author: string (nullable = true)
 |    |    |-- copies_available: long (nullable = true)
 |-- members: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- member_id: string (nullable = true)
 |    |    |-- member_name: string (nullable = true)
 |    |    |-- age: long (nullable = true)
 |    |    |-- books_borrowed: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)



In [13]:
# 3. Given the dataset (/public/trendytech/datasets/train.csv), 
# create a Dataframe using PySpark and perform the following operations :- 
# Drop the columns passenger_name and age from the dataset 
# Count the number of rows after removing duplicates of columnstrain_number and ticket_number 
# Count the number of unique train names.

In [14]:
!hadoop fs -head /public/trendytech/datasets/train.csv

train_number,train_name,seats_available,passenger_name,age,ticket_number,seat_number
123,Express,100,John,25,T123,A1
123,Express,100,Emma,30,T124,B2
456,Superfast,150,Michael,35,T125,C3
456,Superfast,150,Sophia,40,T126,D4
789,Local,50,William,28,T127,E5
789,Local,50,Sophia,32,T128,F6
789,Local,50,Oliver,45,T129,G7


In [15]:
train_df = spark.read.format('csv').option('header','true').option('inferSchema','true').load('/public/trendytech/datasets/train.csv')

In [16]:
train_df.printSchema()

root
 |-- train_number: integer (nullable = true)
 |-- train_name: string (nullable = true)
 |-- seats_available: integer (nullable = true)
 |-- passenger_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- ticket_number: string (nullable = true)
 |-- seat_number: string (nullable = true)



In [17]:
train_df.show(3)

+------------+----------+---------------+--------------+---+-------------+-----------+
|train_number|train_name|seats_available|passenger_name|age|ticket_number|seat_number|
+------------+----------+---------------+--------------+---+-------------+-----------+
|         123|   Express|            100|          John| 25|         T123|         A1|
|         123|   Express|            100|          Emma| 30|         T124|         B2|
|         456| Superfast|            150|       Michael| 35|         T125|         C3|
+------------+----------+---------------+--------------+---+-------------+-----------+
only showing top 3 rows



In [18]:
train_df1 = train_df.drop("passenger_name","age")

In [19]:
train_df1.show(3)

+------------+----------+---------------+-------------+-----------+
|train_number|train_name|seats_available|ticket_number|seat_number|
+------------+----------+---------------+-------------+-----------+
|         123|   Express|            100|         T123|         A1|
|         123|   Express|            100|         T124|         B2|
|         456| Superfast|            150|         T125|         C3|
+------------+----------+---------------+-------------+-----------+
only showing top 3 rows



In [20]:
train_df1.count()

7

In [21]:
train_df2 = train_df1.dropDuplicates(["train_number","ticket_number"])

In [22]:
train_df2.count()

7

In [23]:
train_df.select("train_name").distinct().show()

+----------+
|train_name|
+----------+
|   Express|
|     Local|
| Superfast|
+----------+



In [24]:
train_df.select("train_name").distinct().count()

3

In [25]:
train_df.dropDuplicates(["train_name"]).show()

+------------+----------+---------------+--------------+---+-------------+-----------+
|train_number|train_name|seats_available|passenger_name|age|ticket_number|seat_number|
+------------+----------+---------------+--------------+---+-------------+-----------+
|         123|   Express|            100|          John| 25|         T123|         A1|
|         789|     Local|             50|       William| 28|         T127|         E5|
|         456| Superfast|            150|       Michael| 35|         T125|         C3|
+------------+----------+---------------+--------------+---+-------------+-----------+



In [26]:
# Your task is to read the "sales_data.json" dataset(/public/trendytech/datasets/sales_data.json) using PySpark, 
# utilizing different read modes to handle corrupt records. 
# You need to create aDataframe using pyspark and perform the following operations
# 1. Read the dataset using the "permissive" mode and count the number ofrecords read.
# 2. Read the dataset using the "dropmalformed" mode and display thenumber of malformed records.
# 3. Read the dataset using the "failfast" mode

In [27]:
!hadoop fs -head /public/trendytech/datasets/sales_data.json

{"store_id": 1, "product": "Apple", "quantity": 10, "revenue": 100.0}
{"store_id": 2, "product": "Banana", "quantity": 15, "revenue": 75.0}
{"store_id": 3, "product": "Orange", "quantity": 12, "revenue": 90.0}
{"store_id": 4, "product": "Mango", "quantity": 8, "revenue": 120.0}
{"store_id": 5, "product": "Grape", "quantity": 20, "revenue": 150.0}
{"store_id": 6, "product": "Watermelon", "quantity": 5, "revenue": 50.0}
{"store_id": 7, "product": "Strawberry", "quantity": 18, "revenue": 108.0}
{"store_id": 8, "product": "Pineapple", "quantity": 14, "revenue": 140.0}
{"store_id": 9, "product": "Cherry", "quantity": 7, "revenue": 105.0}
{"store_id": 10, "product": "Pear", "quantity": 9, "revenue": 81.0}
{"store_id": 11, "product": "Blueberry", "quantity": 11, "revenue": 88.0}
{"store_id": 12, "product": "Kiwi", "quantity": 16, "revenue": 128.0}
{"store_id": 13, "product": "Peach", "quantity": 13, "revenue": 91.0}
{"store_id": 14, "product": "Plum", "quantity": 6, "revenue": 54.0}
{"store_i

In [28]:
salesStruct = StructType([
    StructField("store_id",IntegerType()),
    StructField("product",StringType()),
    StructField("quantity",IntegerType()),
    StructField("revenue",DoubleType()),
])

In [29]:
sales_json = spark.read.schema(salesStruct).json('/public/trendytech/datasets/sales_data.json')  #defaule read mode = permissive

### Read modes

In [30]:
sales_json.show(3)

+--------+-------+--------+-------+
|store_id|product|quantity|revenue|
+--------+-------+--------+-------+
|       1|  Apple|      10|  100.0|
|       2| Banana|      15|   75.0|
|       3| Orange|      12|   90.0|
+--------+-------+--------+-------+
only showing top 3 rows



In [31]:
sales_json.printSchema()

root
 |-- store_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- revenue: double (nullable = true)



In [32]:
sales_json.count()

22

In [37]:
schema="store_id integer,product string,quantity integer,revenue double"

In [38]:
sales_json1 = spark.read.format('json').schema(schema).option('mode','dropMalformed').load('/public/trendytech/datasets/sales_data.json')

In [39]:
sales_json1.count()

21

In [40]:
sales_json2 = spark.read.format('json').schema(schema).option('mode','failfast').load('/public/trendytech/datasets/sales_data.json')

In [41]:
sales_json1.count()

21